# Phase III — Karin's 5 Classifiers on Lindsey Features (All 3 Experiments)

Runs Karin's classifier suite (NB, RF, SVM, DNN, DT) on the user-level Lindsey features produced by `phase2_all_experiments.ipynb`. One cell per experiment so each can be run independently.

**Inputs** (in the working directory):
- `phase2_train_exp{1,2,3}.csv`
- `phase2_test_exp{1,2,3}.csv`

**Outputs:**
- `phase3_results.csv` — long-form table (experiment × classifier × recall/precision/F1)

**Notes**
- SVM uses `LinearSVC` (equivalent to `SVC(kernel='linear')` but ~100× faster — original `SVC` ran 2+ days).
- DNN is Karin's Keras ReLU + SGD(lr=1e-3) configuration. CPU-only to avoid cuDNN memory errors.
- NB uses `MultinomialNB` on min-max-scaled features (NB needs non-negative input).

## Cell 1 — Imports and helpers

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # force CPU for Keras; avoids cuDNN OOM

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.naive_bayes    import MultinomialNB
from sklearn.ensemble       import RandomForestClassifier
from sklearn.svm            import LinearSVC
from sklearn.tree           import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

def load_exp(exp_id):
    tr = pd.read_csv(f'phase2_train_exp{exp_id}.csv')
    te = pd.read_csv(f'phase2_test_exp{exp_id}.csv')
    feat_cols = [c for c in tr.columns if c not in ('user_id','label')]
    Xtr, ytr = tr[feat_cols].values, tr['label'].values
    Xte, yte = te[feat_cols].values, te['label'].values
    return Xtr, ytr, Xte, yte, feat_cols

def metrics(y_true, y_pred):
    return {
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }

def fit_predict_all(Xtr, ytr, Xte):
    out = {}

    # NB needs non-negative; use min-max
    mm = MinMaxScaler().fit(Xtr)
    Xtr_mm, Xte_mm = mm.transform(Xtr), mm.transform(Xte)
    nb = MultinomialNB(alpha=1.0).fit(Xtr_mm, ytr)
    out['NB'] = nb.predict(Xte_mm)

    # RF — Karin defaults
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    out['RF'] = rf.predict(Xte)

    # SVM linear
    svm = LinearSVC(C=1.0, random_state=RANDOM_STATE, max_iter=5000).fit(Xtr, ytr)
    out['SVM'] = svm.predict(Xte)

    # DNN — Karin's ReLU + SGD lr=1e-3 architecture
    ss = StandardScaler().fit(Xtr)
    Xtr_ss, Xte_ss = ss.transform(Xtr), ss.transform(Xte)
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    dnn = Sequential([
        Input(shape=(Xtr_ss.shape[1],)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1,  activation='sigmoid'),
    ])
    dnn.compile(optimizer=SGD(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    dnn.fit(Xtr_ss, ytr, epochs=200, batch_size=16, verbose=0)
    out['DNN'] = (dnn.predict(Xte_ss, verbose=0).ravel() >= 0.5).astype(int)

    # DT with GridSearchCV — Karin uses k-fold tuned tree
    skf = StratifiedKFold(n_splits=min(5, max(2, ytr.sum())), shuffle=True, random_state=RANDOM_STATE)
    dt_grid = {'max_depth': [None, 5, 10, 20], 'min_samples_split': [2, 5, 10]}
    dt_gs = GridSearchCV(
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        dt_grid, scoring='f1', cv=skf, n_jobs=-1
    ).fit(Xtr, ytr)
    out['DT'] = dt_gs.predict(Xte)

    return out

def run_experiment(exp_id):
    print(f'\n{"="*70}\n  Experiment {exp_id}\n{"="*70}')
    Xtr, ytr, Xte, yte, feat_cols = load_exp(exp_id)
    print(f'  train: {Xtr.shape} (pos={ytr.sum()})  |  test: {Xte.shape} (pos={yte.sum()})')
    preds = fit_predict_all(Xtr, ytr, Xte)

    rows = []
    for clf, ypred in preds.items():
        m = metrics(yte, ypred)
        rows.append({'experiment': exp_id, 'classifier': clf, **m})
        print(f'  {clf:>4}  recall={m["recall"]:.4f}  precision={m["precision"]:.4f}  f1={m["f1"]:.4f}')
    return pd.DataFrame(rows)

print('Helpers ready')


2026-05-24 10:59:11.147604: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-24 10:59:11.147630: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-24 10:59:11.147648: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-24 10:59:11.153006: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-24 10:59:12.134142: W tensorflow/compiler/

Helpers ready


## Cell 2 — Experiment 1 (~30/70)

In [2]:
res_exp1 = run_experiment(1)
res_exp1


  Experiment 1
  train: (86, 62) (pos=43)  |  test: (194, 62) (pos=97)


/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
2026-05-24 10:59:13.840614: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-05-24 10:59:13.840638: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:168] retrieving CUDA diagnostic information for host: idea-26
2026-05-24 10:59:13.840641: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:175] hostname: idea-26
2026-05-24 10:59:13.840725: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.c

    NB  recall=0.7010  precision=0.6071  f1=0.6507
    RF  recall=0.1340  precision=0.5417  f1=0.2149
   SVM  recall=0.1649  precision=0.4706  f1=0.2443
   DNN  recall=0.6701  precision=0.5963  f1=0.6311
    DT  recall=0.1959  precision=0.4750  f1=0.2774


,experiment,classifier,recall,precision,f1
0,1,NB,0.701031,0.607143,0.650718
1,1,RF,0.134021,0.541667,0.214876
2,1,SVM,0.164948,0.470588,0.244275
3,1,DNN,0.670103,0.596330,0.631068
4,1,DT,0.195876,0.475000,0.277372


## Cell 3 — Experiment 2 (50/50)

In [3]:
res_exp2 = run_experiment(2)
res_exp2


  Experiment 2
  train: (140, 62) (pos=70)  |  test: (140, 62) (pos=70)


/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    NB  recall=0.9000  precision=0.5676  f1=0.6961
    RF  recall=0.2000  precision=0.5600  f1=0.2947
   SVM  recall=0.3429  precision=0.3429  f1=0.3429
   DNN  recall=0.8429  precision=0.5784  f1=0.6860
    DT  recall=0.2143  precision=0.4688  f1=0.2941


,experiment,classifier,recall,precision,f1
0,2,NB,0.900000,0.567568,0.696133
1,2,RF,0.200000,0.560000,0.294737
2,2,SVM,0.342857,0.342857,0.342857
3,2,DNN,0.842857,0.578431,0.686047
4,2,DT,0.214286,0.468750,0.294118


## Cell 4 — Experiment 3 (~70/30)

In [4]:
res_exp3 = run_experiment(3)
res_exp3


  Experiment 3
  train: (198, 62) (pos=99)  |  test: (82, 62) (pos=41)


/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/Ken/anaconda3/envs/SiMAIM_TGIF_2/lib/python3.10/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    NB  recall=0.9268  precision=0.5588  f1=0.6972
    RF  recall=0.1707  precision=0.3684  f1=0.2333
   SVM  recall=0.1220  precision=0.1515  f1=0.1351
   DNN  recall=0.8780  precision=0.5625  f1=0.6857
    DT  recall=0.0732  precision=0.2000  f1=0.1071


,experiment,classifier,recall,precision,f1
0,3,NB,0.926829,0.558824,0.697248
1,3,RF,0.170732,0.368421,0.233333
2,3,SVM,0.121951,0.151515,0.135135
3,3,DNN,0.878049,0.562500,0.685714
4,3,DT,0.073171,0.200000,0.107143


## Cell 5 — Combined results table

In [5]:
all_results = pd.concat([res_exp1, res_exp2, res_exp3], ignore_index=True)
all_results.to_csv('phase3_results.csv', index=False)

pivot_f1 = all_results.pivot(index='classifier', columns='experiment', values='f1') \
                      .reindex(['NB','RF','SVM','DNN','DT'])
pivot_f1.columns = [f'Exp{c} F1' for c in pivot_f1.columns]
print('F1 across experiments (Lindsey features + Karin classifiers):')
print(pivot_f1.round(4).to_string())
print()
print('Saved → phase3_results.csv')


F1 across experiments (Lindsey features + Karin classifiers):
            Exp1 F1  Exp2 F1  Exp3 F1
classifier                           
NB           0.6507   0.6961   0.6972
RF           0.2149   0.2947   0.2333
SVM          0.2443   0.3429   0.1351
DNN          0.6311   0.6860   0.6857
DT           0.2774   0.2941   0.1071

Saved → phase3_results.csv


## Notes on interpreting results

- SVM is expected to be the only classifier stable across all three splits (F1 ≈ 0.69–0.71).
- DT may collapse to all-zero predictions on Experiment 3 (the hardest split for tree splits on this feature space). This is a substantive finding about feature discriminative power, not a bug.
- Compare against Karin's original Phase II + Phase III results to populate the side-by-side comparison table in the thesis.